In [ ]:
from dotenv import load_dotenv 
import nest_asyncio 


: 

In [ ]:
nest_asyncio.apply()

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
OPENROUTER_API_KEY   = os.getenv("OPENROUTER_API_KEY")    
OPENROUTER_GUARD_KEY = os.getenv("OPENROUTER_GUARD_KEY")  
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
API_URL=os.getenv('API_URL','')
AUTH_TOKEN=os.getenv('AUTH_TOKEN','')
MODEL_NAME=os.getenv('MODEL_NAME','')

LLM_MODEL_1=os.getenv("LLM_MODEL_1")
print("Environment check:")
print(f"  OPENROUTER API Key   : {'OK' if OPENROUTER_API_KEY   else 'MISSING'}")
print(f"  OPENROUTER Guard Key : {'OK' if OPENROUTER_GUARD_KEY  else 'MISSING'}")
print(f"  NVIDIA API Key : {'OK' if NVIDIA_API_KEY else 'MISSING'}")
print (f"  LLM MODEL 1:{'OK ' if LLM_MODEL_1 else 'MISSING' }")
print (f"  API_URL :{'OK ' if API_URL  else 'MISSING' }")
print(f"  MODEL_NAME  : {'OK' if MODEL_NAME  else 'MISSING'}")
print (f"   AUTH_TOKEN :{'OK ' if AUTH_TOKEN  else 'MISSING' }")

In [ ]:
from typing import Optional, List, Any
import requests
from langchain_core.language_models.llms import LLM
from nemoguardrails import RailsConfig, LLMRails
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
# ---------------------------------------------------------------------------
# 1. i Wrap  THE   endpoint as a LangChain-compatible LLM
#    (NeMo Guardrails requires a langchain LLM object, not raw requests calls)
# ---------------------------------------------------------------------------


TIMEOUT_SECONDS = 6


class GURARDLLM(LLM):
    """Minimal LangChain LLM wrapper around the   endpoint."""

    temperature: float = 0.3
    max_tokens: int = 400

    @property
    def _llm_type(self) -> str:
        return "LLM"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,  # <-- this was missing
        **kwargs: Any,
    ) -> str:
        payload = {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {AUTH_TOKEN}",
        }
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=TIMEOUT_SECONDS)
            resp.raise_for_status()
            result = resp.json()
            return result["choices"][0]["message"]["content"]
        except Exception as e:
            return f"[ LLM call failed: {e}]"


guard_llm = GURARDLLM()